In [3]:
%%capture
!pip install --force-reinstall numpy scikit-learn fiftyone
!pip install torch torchvision torchaudio sam2

In [ ]:
pip install --upgrade "elyra[all]"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of kfp to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  

# Dataset download 

In [3]:
pwd

'/opt/app-root/src/rhelai-cv-ppe-demo'

In [4]:
%%capture
# PPE dataset download 
dataset_url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
!mkdir -p datasets/construction-ppe
!wget -O dataset.zip $dataset_url
!unzip -o dataset.zip -d datasets/construction-ppe
!rm dataset.zip

--2026-03-15 22:19:23--  https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/77f1f9a8-5399-4310-af88-c1e1154fa0df?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-15T23%3A01%3A08Z&rscd=attachment%3B+filename%3Dconstruction-ppe.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-15T22%3A00%3A51Z&ske=2026-03-15T23%3A01%3A08Z&sks=b&skv=2018-11-09&sig=bwfALXeyERa6RfCi3cVizP15bhiC258gUuhNaeQgigQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzYxNjc2MywibmJmIjoxNzczNjEzMTYzLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjd

In [5]:
%%writefile ./datasets/construction-ppe/dataset.yaml
path: . # dataset root dir
train: images/train # train images (relative to 'path') 1132 images
val: images/val # val images (relative to 'path') 143 images
test: images/test # test images (relative to 'path') 141 images

# Classes
names:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  5: none
  6: Person
  7: no_helmet
  8: no_goggle
  9: no_gloves
  10: no_boots

Writing ./datasets/construction-ppe/dataset.yaml


# Load the dataset

In [6]:
import fiftyone as fo

# Delete existing dataset if it exists
if "construction-ppe" in fo.list_datasets():
    fo.delete_dataset("construction-ppe")

# Load all splits of the YOLO dataset
dataset = fo.Dataset("construction-ppe")

for split in ["train", "val", "test"]:
    dataset.add_dir(
        dataset_dir="datasets/construction-ppe",
        dataset_type=fo.types.YOLOv5Dataset,
        yaml_path="dataset.yaml",
        split=split,
        tags=split,
    )


/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 100% |███████████████| 1132/1132 [3.5s elapsed, 0s remaining, 356.7 samples/s]      
 100% |█████████████████| 143/143 [425.6ms elapsed, 0s remaining, 336.0 samples/s]      
 100% |█████████████████| 141/141 [442.3ms elapsed, 0s remaining, 318.8 samples/s]      


In [7]:
import fiftyone as fo
dataset = fo.load_dataset("construction-ppe")

In [8]:
print(dataset.summary())

Name:        construction-ppe
Media type:  image
Num samples: 1416
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)


In [9]:
sample = dataset.first()
print(sample)

<Sample: {
    'id': '69b730a6b50bdec635332868',
    'media_type': 'image',
    'filepath': '/opt/app-root/src/rhelai-cv-ppe-demo/datasets/construction-ppe/images/train/image100.jpg',
    'tags': ['train'],
    'metadata': None,
    'created_at': datetime.datetime(2026, 3, 15, 22, 20, 22, 169000),
    'last_modified_at': datetime.datetime(2026, 3, 15, 22, 20, 22, 169000),
    'ground_truth': <Detections: {
        'detections': [
            <Detection: {
                'id': '69b730a6b50bdec63533285d',
                'attributes': {},
                'tags': [],
                'label': 'helmet',
                'bounding_box': [0.370235, 0.34187449999999997, 0.101734, 0.082469],
                'mask': None,
                'mask_path': None,
                'confidence': None,
                'index': None,
            }>,
            <Detection: {
                'id': '69b730a6b50bdec63533285e',
                'attributes': {},
                'tags': [],
                'label

In [10]:
# Check a sample's bounding boxes
sample = dataset.first()
print(sample.filepath)
for det in sample.ground_truth.detections[:3]:
    print(f"  {det.label}: {det.bounding_box}")

/opt/app-root/src/rhelai-cv-ppe-demo/datasets/construction-ppe/images/train/image100.jpg
  helmet: [0.370235, 0.34187449999999997, 0.101734, 0.082469]
  helmet: [0.186031, 0.34187449999999997, 0.119266, 0.078141]
  Person: [0.1105, 0.31179700000000005, 0.216234, 0.4565]


In [20]:
#oc port-forward pod/ppe-lab-0 5151:5151 -n ppe-detection-cv
fo.close_app()
session = fo.launch_app(dataset, port=5151, address="0.0.0.0") # , remote=True

In [21]:
session.show()

@dataset{Dalvi_Construction_PPE_Dataset_2025,
    author    = {Mrunmayee Dalvi and Niyati Singh and Sahil Bhingarde and Ketaki Chalke},
    title     = {Construction-PPE: Personal Protective Equipment Detection Dataset},
    month     = {January},
    year      = {2025},
    version   = {1.0.0},
    license   = {AGPL-3.0},
    url       = {https://docs.ultralytics.com/datasets/detect/construction-ppe/},
    publisher = {Ultralytics}
}

In [23]:
import fiftyone as fo
from IPython.display import IFrame, display

# Override with our own IFrame pointing to the proxy URL
proxy_url = "https://data-science-gateway.apps.cluster-5t99k.5t99k.sandbox1248.opentlc.com/notebook/ppe-detection-cv/ppe-lab/proxy/5151/"
display(IFrame(proxy_url, width="100%", height=800))
